# AgroFlow no Google Cloud (sojatech_cloud, arquivado)

**Este projeto está arquivado:** a conta GCP não ficou disponível e o trabalho roda pelo `sojatech_local`. O notebook fica pronto para quando houver créditos.

Roda Document AI nas notas, Vision API nas folhas, treina e avalia um Vertex AI AutoML e compara **Google vs. local** nos mesmos conjuntos de teste. Não precisa de GPU: o trabalho pesado roda nos servidores do Google.

**Custos (preços de referência, confira antes):** Vision e Document AI ficam em centavos para ~100 imagens e 80 páginas. O treino AutoML (8 horas de nó, o mínimo) é o item caro, ~US$ 25–30. A previsão é feita por *batch prediction*, então **nenhum endpoint fica ligado cobrando por hora**.

## 0. Preparar o ambiente

O repositório é privado: crie um token no GitHub (Settings → Developer settings → Fine-grained tokens, só leitura de *Contents* do repositório `agroflow`) e salve-o nos **Secrets do Colab** (ícone de chave na barra lateral) com o nome `GITHUB_TOKEN`, liberando o acesso para este notebook. Se o repositório for público, pule o token.

In [ ]:
import os, re, subprocess

REPO = "AbnerMSaconi/agroflow"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/agroflow"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/agroflow"], check=True)
%cd /content/agroflow/sojatech_cloud
!pip install -q -r requirements.txt

In [ ]:
from IPython.display import Image, Markdown, display


def sh(cmd):
    # Roda um comando de shell, mostra a saída e devolve o stdout (para pegar IDs).
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=os.environ)
    print(p.stdout[-4000:])
    if p.returncode:
        print(p.stderr[-4000:])
        raise RuntimeError(f"falhou: {cmd}")
    return p.stdout


def mostra_figuras(*nomes):
    for n in nomes:
        caminho = f"results/figures/{n}.png"
        if os.path.exists(caminho):
            display(Image(caminho, width=520))

## A. Lado local, para a comparação

A comparação da seção B8 lê os resultados do `sojatech_local` (SQLite). Esta célula roda o projeto local nesta mesma sessão; use GPU se quiser que seja rápido.

In [ ]:
!cd ../sojatech_local && pip install -q -r requirements.txt && python scripts/train_leaf_classifier.py \
    && python scripts/run_batch.py notas | tail -1 && python scripts/run_batch.py folhas | tail -1

## B1. Configuração

- **PROJECT**: ID do projeto. Precisa ser único no mundo todo (ex.: `agroflow-p2-seunome`). Se o projeto já existe, o script só o reutiliza.
- **BILLING_ACCOUNT**: a conta de faturamento dos créditos (a célula seguinte lista as suas).
- **BUDGET_USD**: o orçamento; você recebe e-mail em 25%, 50%, 90% e 100%.

In [ ]:
PROJECT = ""  #@param {type:"string"}
BILLING_ACCOUNT = ""  #@param {type:"string"}
BUDGET_USD = "50"  #@param {type:"string"}
REGION = "us-central1"  #@param {type:"string"}

assert PROJECT, "preencha PROJECT"
BUCKET = f"{PROJECT}-entrada"
os.environ.update(
    PROJECT=PROJECT, REGION=REGION, BUCKET=BUCKET, BUDGET_USD=BUDGET_USD,
    GCP_PROJECT=PROJECT, GOOGLE_CLOUD_PROJECT=PROJECT, GOOGLE_CLOUD_QUOTA_PROJECT=PROJECT,
    GCP_LOCATION="us", VERTEX_REGION=REGION, BQ_TABLE=f"{PROJECT}.agroflow.resultados",
)
if BILLING_ACCOUNT:
    os.environ["BILLING_ACCOUNT"] = BILLING_ACCOUNT

from google.colab import auth
auth.authenticate_user()
!gcloud billing accounts list

## B2. Projeto, orçamento, APIs, bucket e BigQuery

Roda `infra/setup_gcp.sh`. **O orçamento com alerta é criado antes de qualquer serviço pago.** Leva alguns minutos; pode ser rodado de novo sem problema.

In [ ]:
assert os.environ.get("BILLING_ACCOUNT"), "preencha BILLING_ACCOUNT na célula B1 (veja a lista acima)"
sh("bash infra/setup_gcp.sh")

## B3. Document AI — Invoice Parser nas notas

Cria (ou reaproveita) o processador *Invoice Parser* e processa as 80 notas: as 40 de desenvolvimento e as 40 de teste. O parser foi treinado com faturas estrangeiras; o interessante é ver onde ele erra na DANFE, sobretudo no layout que o extrator local nunca viu.

In [ ]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

cliente = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(api_endpoint="us-documentai.googleapis.com"))
parent = cliente.common_location_path(PROJECT, "us")
existentes = [p for p in cliente.list_processors(parent=parent) if p.display_name == "agroflow-invoice"]
proc = existentes[0] if existentes else cliente.create_processor(
    parent=parent, processor=documentai.Processor(type_="INVOICE_PROCESSOR", display_name="agroflow-invoice"))
os.environ["DOCAI_PROCESSOR_ID"] = proc.name.rsplit("/", 1)[-1]
print("DOCAI_PROCESSOR_ID =", os.environ["DOCAI_PROCESSOR_ID"])

!python scripts/run_batch.py notas | tail -3

## B4. Vision API — linha de base nas folhas

*Label detection* nas 100 folhas de teste. A expectativa é ver rótulos genéricos ("leaf", "plant") com score alto e nada que identifique a doença.

In [ ]:
!python scripts/run_batch.py folhas --sem-classificador | tail -1

## B5. Vertex AI AutoML — treino

Sobe as 400 imagens de treino, cria o dataset e dispara o treino com o orçamento mínimo (8 horas de nó). **O treino leva de 1 a 3 horas** e roda nos servidores do Google: se o Colab desconectar, nada se perde; rode a célula B6 depois para recuperar o modelo.

In [ ]:
saida = sh("bash infra/vertex_train.sh upload")
os.environ["DATASET_ID"] = re.search(r"DATASET_ID (\S+)", saida).group(1)

from google.cloud import aiplatform
aiplatform.init(project=PROJECT, location=REGION)
job = aiplatform.AutoMLImageTrainingJob(display_name="folhas-automl",
                                        prediction_type="classification", model_type="CLOUD")
job.run(dataset=aiplatform.ImageDataset(os.environ["DATASET_ID"]), budget_milli_node_hours=8000,
        training_fraction_split=0.8, validation_fraction_split=0.1, test_fraction_split=0.1,
        model_display_name="folhas-milho", sync=False)
job.wait_for_resource_creation()
print("treino iniciado:", job.resource_name)
print("acompanhe em https://console.cloud.google.com/vertex-ai/training/training-pipelines?project=" + PROJECT)

## B6. Recuperar o modelo e a avaliação do Vertex

Rode depois que o treino terminar (também serve numa sessão nova do Colab: rode antes as células 0 e B1). Salva a avaliação do próprio Vertex em `results/vertex_avaliacao.json` para o relatório.

In [ ]:
import json
from google.cloud import aiplatform
from google.protobuf import json_format
aiplatform.init(project=PROJECT, location=REGION)

modelos = aiplatform.Model.list(filter='display_name="folhas-milho"', order_by="create_time desc")
if not modelos:
    for p in aiplatform.AutoMLImageTrainingJob.list(filter='display_name="folhas-automl"'):
        print(p.display_name, p.state)
    raise SystemExit("modelo ainda não existe: o treino não terminou")
modelo = modelos[0]
os.environ["MODEL_ID"] = modelo.name
print("MODEL_ID =", modelo.name)

metricas = json_format.MessageToDict(modelo.get_model_evaluation().gca_resource._pb).get("metrics", {})
os.makedirs("results", exist_ok=True)
with open("results/vertex_avaliacao.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)
print({k: metricas.get(k) for k in ("auPrc", "logLoss")})

## B7. Vertex AI — classificar o teste por *batch prediction*

Classifica as 100 folhas do mesmo teste fixo usado pelo modelo local. O job sobe as máquinas, classifica e desliga sozinho (20–40 min); não sobra nenhum endpoint ligado.

In [ ]:
sh("bash infra/vertex_train.sh batch")

## B8. Avaliação: Google vs. local

Lê os resultados locais (SQLite, da Parte A) e os do Google (BigQuery) e gera tabelas, gráficos e `report/tabelas/comparacao_backends.tex`. Se você abriu uma sessão nova, rode a célula A antes para ter o lado local.

In [ ]:
!python scripts/evaluate.py > /dev/null
texto = open("results/resumo.md", encoding="utf-8").read()
display(Markdown(texto[texto.find("# Google Cloud"):]))
mostra_figuras("notas_acerto_vs_confidence_teste_gcp", "baseline_rotulos_gcp",
               "folhas_matriz_confusao_gcp", "folhas_calibracao_gcp")

## B9. (Opcional) Pipeline orientado a eventos: Cloud Function

Publica a função que dispara quando um arquivo chega em `gs://<bucket>/entrada/`, e testa com uma nota e uma folha. Sem endpoint do Vertex, a folha passa só pela Vision API. Para a demo ao vivo com o classificador, implante o endpoint (`infra/vertex_train.sh deploy`) **e remova logo depois** (`undeploy`).

In [ ]:
sh("bash infra/deploy_function.sh")
!gcloud storage cp ../data/invoices/teste/pdf/nota_001.pdf gs://$BUCKET/entrada/notas/demo_nota.pdf
!gcloud storage cp ../data/leaves/test/ferrugem_comum/ferrugem_comum_000.jpg gs://$BUCKET/entrada/imagens/demo_folha.jpg
import time; time.sleep(60)
!bq query --use_legacy_sql=false "SELECT arquivo, tipo, campo, valor, confidence FROM \`$BQ_TABLE\` WHERE arquivo LIKE 'demo_%' ORDER BY processado_em DESC LIMIT 20"

## B10. Baixar os resultados

Baixa um zip com `results/` e as tabelas e figuras do relatório para você copiar para o repositório local e fazer o commit.

In [ ]:
!zip -qr /content/agroflow_cloud_resultados.zip results ../report/figuras ../report/tabelas
from google.colab import files
files.download("/content/agroflow_cloud_resultados.zip")

## B11. Desligar o que cobra

Remove qualquer endpoint do Vertex que tenha ficado implantado. O modelo, o dataset e o bucket ficam guardados (custo de armazenamento desprezível). **Depois da apresentação**, apague o projeto inteiro descomentando a última linha.

In [ ]:
from google.cloud import aiplatform
aiplatform.init(project=PROJECT, location=REGION)
for ep in aiplatform.Endpoint.list():
    print("removendo implantações de", ep.display_name)
    ep.undeploy_all()
print("endpoints ativos:", [e.display_name for e in aiplatform.Endpoint.list() if e.list_models()])
# !gcloud projects delete $PROJECT --quiet